# Paper 2 workbench: counterfactual studies on Colab

One notebook for every model: `linear`, `mm_capacity`, `mm_competitive`.
Pick the model, the profile and the realizations in **Settings**, then *Runtime → Run all*.

| Profile | Collocation points | Iterations per stage | Use |
|---|---|---|---|
| `smoke` | 300 | 200 + 20 | check that the notebook runs (numbers mean nothing) |
| `fast` | 3,000 | 5,000 + 500 | quick checks and comparisons, a few minutes per study |
| `paper` | 30,000 | 20,000 + 2,000 | paper 1's settings, the final long runs |

Each study is saved to Google Drive as soon as it finishes (`results.json`, the network
weights and its figures). A rerun skips finished studies, so a dropped session just
continues. Changing a loss weight needs a `TAG`, so its results go to their own folder.

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = 'https://github.com/Hachero98/dopamine-PINN-.git'
REPO_REF = 'causal-linear'
REPO_DIR = Path('/content/dopamine-PINN-')
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
SENTINEL = Path('/tmp/paper2_ready_v1')

if IN_COLAB and not SENTINEL.exists():
    rc = os.system('pip -q install "jax[cuda12]==0.10.2" "flax==0.12.5" "optax>=0.2.3" "jaxopt>=0.8" scipy matplotlib')
    if not REPO_DIR.exists():
        rc |= os.system(f'git clone -q -b {REPO_REF} {REPO_URL} {REPO_DIR}')
    rc |= os.system(f'pip -q install --no-deps -e {REPO_DIR}')
    if rc != 0:
        raise RuntimeError('install failed - see the output above')
    SENTINEL.touch()
    print('Installed. Restarting the kernel; then Runtime -> Run all again.')
    os.kill(os.getpid(), 9)

ROOT = REPO_DIR if IN_COLAB else Path.cwd().parent if Path.cwd().name == 'paper2' else Path.cwd()
if IN_COLAB:
    os.system(f'git -C {REPO_DIR} pull -q')          # newest workbench code
sys.path.insert(0, str(ROOT / 'paper2'))
import jax, cf
from cf.analysis import load_runs, compare, print_summary
from cf.plots import plot_study, plot_compare
from IPython.display import Image, display
print('Ready |', jax.devices())

## Settings

In [ ]:
MODEL   = 'linear'            # 'linear', 'mm_capacity', 'mm_competitive'
PROFILE = 'fast'              # 'smoke', 'fast', 'paper'
SEEDS   = '4001-4005'         # realizations: '4001-4005', '4001 4009 4010' or a list (below 4101)
NET_SEED = 1234

W_RES   = None                # residual weight; None = the profile's (3000)
W_DATA  = None                # data weight; None = the profile's (5)
W_INIT  = None                # initial-condition weight; None = the profile's (1)
T_POWER = None                # early-time collocation t = T u^p; None = uniform (1)
TAG     = ''                  # required when a setting is changed, e.g. 'wres3e4'

USE_DRIVE = True

prof = cf.PROFILES[PROFILE]
if isinstance(SEEDS, str):
    SEEDS = [s for part in SEEDS.replace(',', ' ').split()
             for s in (range(int(part.split('-')[0]), int(part.split('-')[1]) + 1) if '-' in part else [int(part)])]
SEEDS = [int(s) for s in SEEDS]
changes = {k: v for k, v in (('w_res', W_RES), ('w_data', W_DATA), ('w_init', W_INIT), ('t_power', T_POWER))
           if v is not None}
assert TAG or not changes, 'set TAG when changing a setting'
if TAG or changes:
    prof = prof.replace(name=f'{PROFILE}_{TAG}', **changes)
assert max(SEEDS) < 4101

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/paper2_runs')
else:
    OUT = ROOT / 'paper2' / 'runs'
print(f'{MODEL} | {prof.name} | {len(SEEDS)} realizations -> {OUT}')
print(f'collocation {prof.n_domain}, Adam {prof.adam_iters} + L-BFGS {prof.lbfgs_iters} per stage, weights {prof.weights}, t_power {prof.t_power}')

## Run (resumes where it stopped)

In [ ]:
for i, s in enumerate(SEEDS, 1):
    print(f'[{i}/{len(SEEDS)}]', end=' ')
    cf.run_study(MODEL, prof, obs_seed=s, net_seed=NET_SEED, out=OUT)
    d = cf.study.study_dir(OUT, MODEL, prof.name, s, NET_SEED)
    if not (d / 'effect_fields.png').exists():
        plot_study(d)

## Summary: PINN against the reference on the same records, and against the bound

RMSE over realizations (%, log scale for parameters). `ratio` = PINN / reference; a ratio
whose interval includes 1 means the PINN is as good as the reference, i.e. at the
information limit of the design.

In [ ]:
rows = load_runs(OUT, MODEL, prof.name, NET_SEED)
print_summary(compare(rows))
fig_path = OUT / MODEL / prof.name / 'compare.png'
plot_compare(rows, fig_path)
display(Image(filename=str(fig_path)))

## One study in detail

In [ ]:
SHOW = SEEDS[0]
d = cf.study.study_dir(OUT, MODEL, prof.name, SHOW, NET_SEED)
for name in ('effect_summaries', 'effect_fields', 'parameters'):
    display(Image(filename=str(d / f'{name}.png')))

In [ ]:
if IN_COLAB:   # download everything of this model and profile as one zip
    import shutil
    from google.colab import files
    z = shutil.make_archive(f'/content/{MODEL}_{prof.name}', 'zip', OUT / MODEL / prof.name)
    files.download(z)